# 13 - Where did the harness API go? (CPU, 0 GPU-h)

P003 died on `from swegemma.models.discovery import validate_single_declared_model`,
which worked in P002 on 2026-10-07. The competition **republished the wheelhouse
on 2026-10-09 14:43**, between the two runs.

This kernel installs **nothing**. It opens the wheels as zip archives and reports:

1. which `swegemma` / `adk_submission` distributions are present, and their versions;
2. every module inside `swegemma/models/`;
3. every archive member, in any wheel, that defines or mentions
   `validate_single_declared_model`;
4. the same for the competition directory, in case the harness ships there too.

The output is the fix: it names the module that now holds the symbol, which goes
straight into `scripts/session_runner.py` `start_server()` and into the eval
kernel's preflight gate.

**Do not add a GPU to this kernel.** It needs none, and a CPU kernel costs no
GPU quota.

In [ ]:
import json
import re
import sys
import zipfile
from pathlib import Path

WANTED = 'validate_single_declared_model'
CANDIDATES = ('swegemma', 'adk_submission')

def say(*a):
    print(*a, flush=True)

roots = [Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse'),
         Path('/kaggle/input/competitions/gemma-4-developer-agent')]
for r in roots:
    say('root', r, 'exists' if r.is_dir() else 'MISSING')

say('')
say('=' * 70)
say('1. distributions present')
say('=' * 70)

wheels = []
for r in roots:
    if r.is_dir():
        wheels += sorted(r.rglob('*.whl'))
say('total wheels visible:', len(wheels))
for w in wheels:
    if any(c in w.name.lower() for c in CANDIDATES):
        say('  ', w.name, '  <-', w.parent)

say('')
say('=' * 70)
say('2. contents of swegemma/models/ and anything named discovery')
say('=' * 70)

def members(w):
    try:
        with zipfile.ZipFile(w) as z:
            return z.namelist()
    except Exception as exc:
        say('  cannot read', w.name, type(exc).__name__, exc)
        return []

for w in wheels:
    if not any(c in w.name.lower() for c in CANDIDATES):
        continue
    names = members(w)
    say('--', w.name, '(' + str(len(names)), 'members)')
    for n in names:
        if '/models/' in n and n.endswith('.py'):
            say('     models:', n)
    for n in names:
        if 'discovery' in n.lower():
            say('     discovery:', n)

say('')
say('=' * 70)
say('3. which module defines ' + WANTED + '?')
say('=' * 70)

# Scan EVERY wheel, not just the two named ones: the symbol may have moved to a
# different distribution entirely, which is exactly the case a name-based search
# would miss.
pat_def = re.compile(rb'def\s+' + WANTED.encode() + rb'\s*\(')
hits_def, hits_mention = [], []
for w in wheels:
    try:
        with zipfile.ZipFile(w) as z:
            for n in z.namelist():
                if not n.endswith('.py'):
                    continue
                try:
                    blob = z.read(n)
                except Exception:
                    continue
                if pat_def.search(blob):
                    hits_def.append((w.name, n))
                elif WANTED.encode() in blob:
                    hits_mention.append((w.name, n))
    except Exception:
        continue

say('DEFINED in:')
for wn, n in hits_def or []:
    say('   ', wn, '->', n)
if not hits_def:
    say('    NOTHING DEFINES IT. The symbol is gone, not moved -- find what')
    say('    replaced it by reading the new models/__init__.py below.')

say('mentioned in (not a definition):')
for wn, n in hits_mention[:25]:
    say('   ', wn, '->', n)

say('')
say('=' * 70)
say('4. the new swegemma/models/__init__.py, verbatim')
say('=' * 70)

for w in wheels:
    if 'swegemma' not in w.name.lower():
        continue
    with zipfile.ZipFile(w) as z:
        for n in z.namelist():
            if n.endswith('swegemma/models/__init__.py'):
                say('--', w.name, n)
                say(z.read(n).decode('utf-8', 'replace')[:6000])

say('')
say('=' * 70)
say('5. adk_submission/discovery.py -- the module that APPEARED')
say('=' * 70)

# P004 round 1 found that swegemma/models/discovery.py is gone while
# adk_submission GAINED a discovery.py, and that nothing anywhere defines
# validate_single_declared_model. So the function was RENAMED, not relocated,
# and the replacement has to be read off the new module's public surface.
for w in wheels:
    if 'adk_submission' not in w.name.lower():
        continue
    with zipfile.ZipFile(w) as z:
        for n in z.namelist():
            if n.endswith('discovery.py') or n.endswith('adk_submission/__init__.py'):
                body = z.read(n).decode('utf-8', 'replace')
                say('')
                say('--- ' + w.name + ' :: ' + n + ' (' + str(len(body)) + ' B) ---')
                for line in body.splitlines():
                    if re.match(r'\s*(def |class |__all__|from |import )', line):
                        say('   ' + line.rstrip()[:160])

say('')
say('=' * 70)
say('6. anything that validates a declared model, by behaviour not by name')
say('=' * 70)

for w in wheels:
    if not any(c in w.name.lower() for c in CANDIDATES):
        continue
    with zipfile.ZipFile(w) as z:
        for n in z.namelist():
            if not n.endswith('.py'):
                continue
            body = z.read(n).decode('utf-8', 'replace')
            for m in re.finditer(r'^\s*def\s+(\w*(declared|model|validate)\w*)\s*\(',
                                 body, re.M | re.I):
                say('   ', w.name, n, '->', m.group(1))

say('')
say('=' * 70)
say('7. the replacement, in full: discover_declared_models + validate_directory')
say('=' * 70)

# P004 round 2 identified these two as the successors to the deleted
# validate_single_declared_model. session_runner needs the exact signature and
# return shape to call them, so print each def block verbatim.
TARGETS = ('discover_declared_models', 'validate_directory')
for w in wheels:
    if 'adk_submission' not in w.name.lower():
        continue
    with zipfile.ZipFile(w) as z:
        for n in z.namelist():
            if not n.endswith('discovery.py'):
                continue
            src = z.read(n).decode('utf-8', 'replace').splitlines()
            for t in TARGETS:
                start = next((k for k, l in enumerate(src)
                              if l.startswith('def ' + t)), None)
                if start is None:
                    say('   ' + t + ': not found in ' + n)
                    continue
                # Run to the next top-level def/class/decorator. Testing for
                # "column 0" instead stops at the signature's own closing
                # paren, which is why round 3 printed only three lines.
                end = next((k for k in range(start + 1, len(src))
                            if src[k].startswith(('def ', 'class ', '@'))), len(src))
                say('')
                say('--- ' + t + '  (' + n + ' lines ' + str(start + 1)
                    + '-' + str(end) + ') ---')
                for l in src[start:min(end, start + 70)]:
                    say('   ' + l.rstrip()[:170])
        # And how __init__ re-exports it, which is the import path to use.
        for n in z.namelist():
            if n.endswith('adk_submission/__init__.py'):
                body = z.read(n).decode('utf-8', 'replace')
                i = body.find('from .discovery import')
                say('')
                say('--- re-export block from adk_submission/__init__.py ---')
                say(body[i:i + 400] if i >= 0 else '(no .discovery re-export found)')

say('')
say('=== COPY FROM HERE ===')
print(json.dumps({
    'wheels_visible': len(wheels),
    'defined_in': hits_def,
    'mentioned_in': hits_mention[:25],
}, indent=1))
say('=== TO HERE ===')